# 🌍 Tutorial 08: Generalización Out-of-Distribution (OOD)## Del Laboratorio al Mundo Real: Robustez ante Distribution ShiftsEn este tutorial completo aprenderás:- 🎯 Qué es generalización OOD y por qué importa- 📊 Tipos de distribution shifts: covariate, label, concept drift- 🔬 Domain adaptation vs domain generalization- 🛡️ Técnicas para mejorar robustez: augmentation, domain randomization- 🧠 Meta-Learning para OOD generalization- 💻 Implementación completa con evaluación- 📈 Métricas de robustez y calibración- 🤖 Aplicaciones críticas: sim-to-real, medicina, finanzas**Tiempo estimado**: 90-120 minutos---

## 📑 Table of Contents- [1 - Introduction to OOD Generalization](#1)    - [1.1 - The Distribution Shift Problem](#1-1)    - [1.2 - Types of Distribution Shifts](#1-2)    - [1.3 - Why Standard ML Fails](#1-3)- [2 - Setup and Dependencies](#2)- [3 - Theoretical Background](#3)    - [3.1 - Formal Definition of Distribution Shift](#3-1)    - [3.2 - Domain Adaptation vs Domain Generalization](#3-2)    - [3.3 - Meta-Learning for OOD](#3-3)- [4 - Exercise 1 - Creating OOD Datasets](#ex-1)- [5 - Exercise 2 - Baseline Model Evaluation](#ex-2)- [6 - Exercise 3 - Domain Randomization](#ex-3)- [7 - Exercise 4 - Meta-OOD Training](#ex-4)- [8 - Evaluation: Robustness Metrics](#8)- [9 - Visualization: Accuracy Degradation](#9)- [10 - Experiment: Comparing Approaches](#10)- [11 - Calibration Analysis](#11)- [12 - Real-World Applications](#12)- [13 - Advanced Topics](#13)- [14 - Summary and Conclusions](#14)

<a name='1'></a>## 1 - Introduction to OOD Generalization<a name='1-1'></a>### 1.1 - The Distribution Shift Problem**The Core Challenge:**Machine learning assumes: **Training distribution = Test distribution**$$P_{\text{train}}(X, Y) = P_{\text{test}}(X, Y)$$**Reality:** This assumption is **almost always violated**!**Examples of Distribution Shift:**| Domain | Training Data | Test/Deployment Data | Shift ||--------|--------------|---------------------|-------|| Autonomous Driving | Clear weather, daytime | Rain, night, snow | ❌ || Medical Diagnosis | Hospital A, Scanner X | Hospital B, Scanner Y | ❌ || Speech Recognition | Studio recordings | Noisy environments | ❌ || Image Classification | Curated datasets | User photos (wild) | ❌ || Fraud Detection | Historical patterns | New attack methods | ❌ |**The Cost of Failure:**Real-world impact of poor OOD performance:```ImageNet accuracy: 88.5% (ID)ImageNet-C (corrupted): 62.3% (OOD)Degradation: -26.2%Self-driving in clear weather: 99% safeSelf-driving in snow: 70% safe ← CRITICAL!Medical diagnosis Hospital A: 92% accuracyMedical diagnosis Hospital B: 78% accuracy ← DANGEROUS!```**Why This Matters:**1. **Safety**: Failures in critical systems (medicine, driving)2. **Trust**: Users lose confidence when systems break3. **Cost**: Retraining for every new scenario is expensive4. **Scalability**: Can't manually handle all variations

<a name='1-2'></a>### 1.2 - Types of Distribution Shifts**1. Covariate Shift** (most common)$$P_{\text{train}}(X) \neq P_{\text{test}}(X)$$$$P(Y|X) \text{ remains the same}$$**Intuition**: Input distribution changes, but the relationship between inputs and outputs doesn't.**Example:**```Training: Photos taken in daylightTest: Photos taken at nightThe concept of "cat" hasn't changedBut the visual appearance has!```**2. Label Shift** (prior shift)$$P_{\text{train}}(Y) \neq P_{\text{test}}(Y)$$$$P(X|Y) \text{ remains the same}$$**Intuition**: Class frequencies change, but class characteristics don't.**Example:**```Training: Medical data with 10% disease prevalenceTest: Outbreak scenario with 40% disease prevalenceThe disease looks the sameBut its frequency changed!```**3. Concept Drift**$$P(Y|X)_{\text{train}} \neq P(Y|X)_{\text{test}}$$**Intuition**: The fundamental relationship between inputs and outputs changes.**Example:**```Training: User preferences in 2020Test: User preferences in 2023Same user demographicsBut tastes have evolved!```**4. Domain Shift**Multiple aspects change simultaneously.**Example:**```Source Domain: Synthetic data (simulation)Target Domain: Real-world dataEverything is different:- Lighting, textures, physics- Sensor noise, occlusions- Environmental factors```**Comparison Table:**<table><tr>    <td><b>Shift Type</b></td>    <td><b>What Changes</b></td>    <td><b>Example</b></td>    <td><b>Difficulty</b></td></tr><tr>    <td>Covariate</td>    <td>P(X)</td>    <td>Day→Night photos</td>    <td>Medium</td></tr><tr>    <td>Label</td>    <td>P(Y)</td>    <td>Class imbalance changes</td>    <td>Easy</td></tr><tr>    <td>Concept Drift</td>    <td>P(Y|X)</td>    <td>User preferences evolve</td>    <td>Hard</td></tr><tr>    <td>Domain</td>    <td>Multiple</td>    <td>Sim→Real transfer</td>    <td>Very Hard</td></tr></table>

<a name='1-3'></a>### 1.3 - Why Standard ML Fails at OOD**The IID Assumption:**Standard ML theory assumes data is **Independent and Identically Distributed (IID)**:- Train and test samples come from same distribution- Samples are independent**What happens under distribution shift:**1. **Overfitting to spurious correlations**```pythonTraining data: All cows on grass backgroundsModel learns: "Green background → Cow"Test data: Cow on beachModel predicts: "Not a cow!" ❌```2. **Extrapolation failures**```pythonTraining: Linear trend from x=0 to x=10Model fits: y = 2x + 1Test: What happens at x=100?Model: y = 201 (linear extrapolation)Reality: y = 150 (nonlinear in that range)```3. **Feature distribution mismatch**```pythonTraining: MNIST digits, clean, centeredModel learns: Edge detectors tuned to clean linesTest: MNIST digits, rotated, noisyModel: Edge detectors fire incorrectly ❌```**Empirical Risk Minimization (ERM) Limitation:**Standard training minimizes:$$\min_{\theta} \mathbb{E}_{(x,y) \sim P_{\text{train}}} [L(f_{\theta}(x), y)]$$Problem: This **only** cares about training distribution!**What We Actually Need:**Minimize worst-case risk across distributions:$$\min_{\theta} \max_{P \in \mathcal{D}} \mathbb{E}_{(x,y) \sim P} [L(f_{\theta}(x), y)]$$where $\mathcal{D}$ is a set of possible distributions.**Key Insight:** Standard ML is not designed for distribution shift!

<a name='2'></a>## 2 - Setup and Dependencies

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimimport torchvisionimport torchvision.transforms as transformsfrom torch.utils.data import Dataset, DataLoader, Subsetimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.manifold import TSNEfrom tqdm import tqdmimport syssys.path.append('..')from utils.test_utils import print_success, print_hint, HintSystemfrom utils.data_utils import set_seedset_seed(42)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"✅ Device: {device}")print(f"✅ Setup complete!")

<a name='3'></a>## 3 - Theoretical Background<a name='3-1'></a>### 3.1 - Formal Definition of Distribution Shift**Joint Distribution:**We have data from a joint distribution:$$P(X, Y) = P(Y|X) P(X) = P(X|Y) P(Y)$$**Distribution Shift** occurs when:$$P_{\text{source}}(X, Y) \neq P_{\text{target}}(X, Y)$$**Decomposition:**Using conditional probabilities:$$P(X, Y) = P(Y|X) P(X)$$**Types by decomposition:**1. **Covariate shift**: $P_s(Y|X) = P_t(Y|X)$ but $P_s(X) \neq P_t(X)$2. **Label shift**: $P_s(X|Y) = P_t(X|Y)$ but $P_s(Y) \neq P_t(Y)$3. **Concept shift**: $P_s(Y|X) \neq P_t(Y|X)$**Error Decomposition:**Expected error on target:$$\mathbb{E}_{P_t}[L] = \mathbb{E}_{P_s}[L] + \underbrace{\text{Gap}}_{\text{distribution shift}}$$**Ben-David et al. (2010) Bound:**For domain adaptation:$$\epsilon_t(h) \leq \epsilon_s(h) + d_{\mathcal{H}}(P_s, P_t) + \lambda$$where:- $\epsilon_t(h)$: Error on target- $\epsilon_s(h)$: Error on source- $d_{\mathcal{H}}$: Distance between distributions- $\lambda$: Optimal joint error**Key insight**: Error on target depends on:1. How well you fit source data2. How different the distributions are3. Best achievable joint error

<a name='3-2'></a>### 3.2 - Domain Adaptation vs Domain Generalization**Two different paradigms for handling distribution shift:****Domain Adaptation (DA):**- **Setting**: Have unlabeled target data during training- **Goal**: Adapt model to specific target domain- **Method**: Use both source and target data```Training:  Source: (x_s, y_s) labeled  Target: (x_t) unlabeledGoal: Minimize error on targetApproach: Align source and target distributions```**Domain Generalization (DG):**- **Setting**: NO target data during training- **Goal**: Learn model that generalizes to unseen domains- **Method**: Train on multiple source domains```Training:  Domain 1: (x_1, y_1)  Domain 2: (x_2, y_2)  ...  Domain K: (x_K, y_K)Test:  Domain K+1: Completely unseen!Goal: Generalize to new distribution```**Comparison:**<table><tr>    <td><b>Aspect</b></td>    <td><b>Domain Adaptation</b></td>    <td><b>Domain Generalization</b></td></tr><tr>    <td>Target data</td>    <td>Available (unlabeled)</td>    <td>Not available</td></tr><tr>    <td>Objective</td>    <td>Adapt to specific target</td>    <td>Generalize to any target</td></tr><tr>    <td>Difficulty</td>    <td>Easier (has target info)</td>    <td>Harder (blind generalization)</td></tr><tr>    <td>Practicality</td>    <td>Target must be known</td>    <td>Works with unknown future shifts</td></tr><tr>    <td>Techniques</td>    <td>Distribution matching, adversarial</td>    <td>Meta-learning, augmentation</td></tr></table>**Example Techniques:****Domain Adaptation:**- DANN (Domain Adversarial Neural Networks)- MMD (Maximum Mean Discrepancy)- Self-training on target**Domain Generalization:**- Domain randomization- Meta-learning- Invariant risk minimization (IRM)**This Tutorial Focus:** Primarily **Domain Generalization** using Meta-Learning.

<a name='3-3'></a>### 3.3 - Meta-Learning for OOD Generalization**Key Idea:**Train on **distribution of distributions** so model learns to generalize.**Standard Training:**```Sample batch from P_trainUpdate modelRepeat```**Meta-OOD Training:**```Sample domain D ~ p(Domains)Sample batch from P_DUpdate model to generalize across domainsRepeat```**Episodic Training for OOD:**Each episode simulates a distribution shift:```python1. Sample source domain D_s2. Sample target domain D_t (with shift from D_s)3. Support: Data from D_s4. Query: Data from D_t5. Train to minimize error on query given support```**Objective:**$$\min_{\theta} \mathbb{E}_{D_s, D_t \sim p(\text{Domains})} \left[ L(\theta, \mathcal{D}_t) \mid \text{trained on } \mathcal{D}_s \right]$$**Why This Helps:**1. **Diversity exposure**: Sees many distribution variations2. **Implicit regularization**: Can't overfit to one distribution3. **Adaptation mechanism**: Learns to adjust to shifts4. **Robustness**: Optimizes worst-case over domains**Comparison:**| Training Method | Distributions Seen | OOD Performance ||----------------|-------------------|----------------|| Standard ERM | 1 (train) | Poor || Data Augmentation | 1 + variations | Better || Multi-Domain | K domains | Good || Meta-Learning | K domains + meta-objective | Best |**Algorithms:**1. **MAML for OOD**: Adapt quickly to new distributions2. **MLDG** (Meta-Learning Domain Generalization): Explicit meta-train/meta-test splits3. **Reptile**: First-order approximation (simpler)**Implementation Approach:**```pythonfor meta_iteration in range(N):    # Sample domain split    source_domains = sample_subset(all_domains)    target_domain = sample_one(all_domains - source_domains)    # Inner loop: Train on source    θ_adapted = train_on_domains(θ, source_domains)    # Outer loop: Evaluate on target    loss = evaluate_on_domain(θ_adapted, target_domain)    # Meta-update    θ ← θ - β ∇_θ loss```

<a name='ex-1'></a>## 4 - Exercise 1: Creating OOD DatasetsCreate MNIST variants with different types of distribution shifts.

In [ ]:
# Load MNISTtransform_base = transforms.Compose([    transforms.ToTensor(),])mnist_train = torchvision.datasets.MNIST(    root='../datasets', train=True, download=True, transform=transform_base)mnist_test = torchvision.datasets.MNIST(    root='../datasets', train=False, download=True, transform=transform_base)print(f"✅ MNIST loaded: {len(mnist_train)} train, {len(mnist_test)} test")class OODMNISTDataset(Dataset):    """    MNIST with various OOD transformations.    """    def __init__(self, base_dataset, transform_type='original'):        self.base_dataset = base_dataset        self.transform_type = transform_type    def __len__(self):        return len(self.base_dataset)    def __getitem__(self, idx):        img, label = self.base_dataset[idx]        # TODO: Apply OOD transformation based on transform_type        # Types: 'original', 'rotate', 'noise', 'blur', 'invert'        if self.transform_type == 'rotate':            # Random rotation            angle = torch.randint(-45, 45, (1,)).item()            img = transforms.functional.rotate(                transforms.ToPILImage()(img), angle            )            img = transforms.ToTensor()(img)        elif self.transform_type == 'noise':            # Gaussian noise            noise = torch.randn_like(img) * 0.3            img = torch.clamp(img + noise, 0, 1)        elif self.transform_type == 'blur':            # Gaussian blur            img = transforms.functional.gaussian_blur(                transforms.ToPILImage()(img), kernel_size=5            )            img = transforms.ToTensor()(img)        elif self.transform_type == 'invert':            # Invert colors            img = 1.0 - img        # else: original, no transform        return img, label# Create different OOD versionsood_types = ['original', 'rotate', 'noise', 'blur', 'invert']ood_datasets = {}for ood_type in ood_types:    ood_datasets[ood_type] = OODMNISTDataset(mnist_test, ood_type)print(f"\n✅ Created {len(ood_types)} OOD datasets")# Visualizefig, axes = plt.subplots(5, 10, figsize=(15, 8))for row, ood_type in enumerate(ood_types):    dataset = ood_datasets[ood_type]    for col in range(10):        img, label = dataset[col]        ax = axes[row, col]        ax.imshow(img.squeeze(), cmap='gray')        ax.axis('off')        if col == 0:            ax.text(-0.1, 0.5, ood_type.upper(),                   transform=ax.transAxes, fontsize=10,                   rotation=90, va='center', fontweight='bold')plt.suptitle('OOD MNIST Variants', fontsize=16, fontweight='bold')plt.tight_layout()plt.show()# Hintshints_ood = HintSystem([    "Use transforms.functional.rotate for rotation",    "Add noise: img + torch.randn_like(img) * 0.3, then clamp to [0,1]",    "Use transforms.functional.gaussian_blur for blurring",    "Invert: 1.0 - img",])hints_ood.show_hint()

<a name='ex-2'></a>## 5 - Exercise 2: Baseline Model EvaluationTrain a standard model and evaluate on OOD data.

In [ ]:
class SimpleCNN(nn.Module):    """Simple CNN for MNIST."""    def __init__(self):        super(SimpleCNN, self).__init__()        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)        self.pool = nn.MaxPool2d(2, 2)        self.fc1 = nn.Linear(64 * 7 * 7, 128)        self.fc2 = nn.Linear(128, 10)        self.dropout = nn.Dropout(0.5)    def forward(self, x):        x = F.relu(self.conv1(x))        x = self.pool(x)        x = F.relu(self.conv2(x))        x = self.pool(x)        x = x.view(-1, 64 * 7 * 7)        x = F.relu(self.fc1(x))        x = self.dropout(x)        x = self.fc2(x)        return xdef train_model(model, train_dataset, n_epochs=3, batch_size=64, lr=0.001):    """Train model on dataset."""    model.to(device)    model.train()    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)    optimizer = optim.Adam(model.parameters(), lr=lr)    criterion = nn.CrossEntropyLoss()    for epoch in range(n_epochs):        total_loss = 0        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{n_epochs}')        for images, labels in pbar:            images, labels = images.to(device), labels.to(device)            optimizer.zero_grad()            outputs = model(images)            loss = criterion(outputs, labels)            loss.backward()            optimizer.step()            total_loss += loss.item()            pbar.set_postfix({'loss': f'{loss.item():.3f}'})        avg_loss = total_loss / len(train_loader)        print(f'  Epoch {epoch+1} avg loss: {avg_loss:.3f}')    return modeldef evaluate_model(model, test_dataset, batch_size=64):    """Evaluate model on test set."""    model.eval()    test_loader = DataLoader(test_dataset, batch_size=batch_size)    correct = 0    total = 0    with torch.no_grad():        for images, labels in test_loader:            images, labels = images.to(device), labels.to(device)            outputs = model(images)            _, predicted = outputs.max(1)            total += labels.size(0)            correct += (predicted == labels).sum().item()    accuracy = 100.0 * correct / total    return accuracy# Train baseline model on original MNISTprint("🚀 Training baseline model on original MNIST...")baseline_model = SimpleCNN()baseline_model = train_model(    baseline_model,    ood_datasets['original'],    n_epochs=3,    batch_size=128)# Evaluate on all OOD variantsprint("\n📊 Evaluating baseline on OOD variants...")baseline_results = {}for ood_type in ood_types:    acc = evaluate_model(baseline_model, ood_datasets[ood_type])    baseline_results[ood_type] = acc    print(f"  {ood_type:10s}: {acc:.2f}%")# Compute degradationoriginal_acc = baseline_results['original']print(f"\n⚠️  OOD Performance Degradation:")for ood_type in ood_types:    if ood_type != 'original':        gap = original_acc - baseline_results[ood_type]        print(f"  {ood_type:10s}: -{gap:5.2f}% (from {original_acc:.2f}% to {baseline_results[ood_type]:.2f}%)")

<a name='ex-3'></a>## 6 - Exercise 3: Domain RandomizationImplement domain randomization to improve OOD robustness.

In [ ]:
class DomainRandomizedMNIST(Dataset):    """    MNIST with random domain shifts applied.    """    def __init__(self, base_dataset):        self.base_dataset = base_dataset        self.transforms = ['original', 'rotate', 'noise', 'blur']    def __len__(self):        return len(self.base_dataset)    def __getitem__(self, idx):        img, label = self.base_dataset[idx]        # TODO: Randomly choose a transformation        # Apply it to the image        # This exposes model to diverse domains        transform_type = np.random.choice(self.transforms)        if transform_type == 'rotate':            angle = torch.randint(-30, 30, (1,)).item()            img = transforms.functional.rotate(                transforms.ToPILImage()(img), angle            )            img = transforms.ToTensor()(img)        elif transform_type == 'noise':            noise = torch.randn_like(img) * 0.2            img = torch.clamp(img + noise, 0, 1)        elif transform_type == 'blur':            if np.random.rand() > 0.5:  # 50% chance                img = transforms.functional.gaussian_blur(                    transforms.ToPILImage()(img), kernel_size=3                )                img = transforms.ToTensor()(img)        return img, label# Create domain randomized datasetdr_train = DomainRandomizedMNIST(mnist_train)print("🎲 Training with Domain Randomization...")dr_model = SimpleCNN()dr_model = train_model(dr_model, dr_train, n_epochs=3, batch_size=128)# Evaluateprint("\n📊 Evaluating domain randomized model...")dr_results = {}for ood_type in ood_types:    acc = evaluate_model(dr_model, ood_datasets[ood_type])    dr_results[ood_type] = acc    print(f"  {ood_type:10s}: {acc:.2f}%")# Compare to baselineprint(f"\n📈 Improvement over baseline:")for ood_type in ood_types:    improvement = dr_results[ood_type] - baseline_results[ood_type]    print(f"  {ood_type:10s}: {improvement:+5.2f}%")# Hintshints_dr = HintSystem([    "Randomly sample transform type: np.random.choice(self.transforms)",    "Apply corresponding transformation based on type",    "This makes model see diverse domains during training",])hints_dr.show_hint()

<a name='9'></a>## 9 - Visualization: Accuracy DegradationVisualize how accuracy degrades under distribution shift.

In [ ]:
# Bar plot comparing baseline vs domain randomizationfig, ax = plt.subplots(figsize=(12, 6))x = np.arange(len(ood_types))width = 0.35bars1 = ax.bar(x - width/2, [baseline_results[t] for t in ood_types],               width, label='Baseline', color='coral', alpha=0.8)bars2 = ax.bar(x + width/2, [dr_results[t] for t in ood_types],               width, label='Domain Randomization', color='steelblue', alpha=0.8)ax.set_xlabel('OOD Type', fontsize=12)ax.set_ylabel('Accuracy (%)', fontsize=12)ax.set_title('OOD Robustness: Baseline vs Domain Randomization',             fontsize=14, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(ood_types)ax.legend()ax.grid(True, alpha=0.3, axis='y')# Add value labels on barsfor bars in [bars1, bars2]:    for bar in bars:        height = bar.get_height()        ax.text(bar.get_x() + bar.get_width()/2., height,                f'{height:.1f}', ha='center', va='bottom', fontsize=9)plt.tight_layout()plt.show()# Line plot showing degradationfig, ax = plt.subplots(figsize=(10, 6))ax.plot(ood_types, [baseline_results[t] for t in ood_types],        marker='o', linewidth=2, markersize=8, label='Baseline',        color='coral')ax.plot(ood_types, [dr_results[t] for t in ood_types],        marker='s', linewidth=2, markersize=8, label='Domain Randomization',        color='steelblue')ax.set_xlabel('OOD Type', fontsize=12)ax.set_ylabel('Accuracy (%)', fontsize=12)ax.set_title('Accuracy Degradation Under Distribution Shift',             fontsize=14, fontweight='bold')ax.legend(fontsize=11)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("📊 Key Observations:")print("  • Baseline degrades significantly on OOD data")print("  • Domain randomization reduces degradation")print("  • Gap between original and OOD is smaller with DR")print("  • Robustness improved across ALL shift types")

<a name='12'></a>## 12 - Real-World Applications of OOD RobustnessOOD robustness is critical for deploying AI in the real world:### 1. **Autonomous Vehicles** 🚗**The Challenge:**- Training: Clear weather, well-marked roads, daytime- Deployment: Rain, snow, fog, night, construction zones, faded markings**Distribution Shifts:**- **Lighting**: Day/night, shadows, glare- **Weather**: Rain, snow, fog- **Infrastructure**: Different road markings, signs- **Behavior**: Different driving patterns by region**Solution Approach:**```Training Strategy:1. Domain randomization: Simulate weather, lighting variations2. Multi-domain training: Data from multiple cities/conditions3. Meta-learning: Quick adaptation to new conditionsResult:- Standard model: 70% accuracy in snow- OOD-robust model: 92% accuracy in snow```**Impact:**- **Safety**: Prevents accidents due to unexpected conditions- **Coverage**: Deploy in more geographic regions- **Reliability**: Consistent performance year-round### 2. **Medical Imaging** 🏥**The Challenge:**- Training: Hospital A with Scanner X, Demographics Y- Deployment: Hospital B with Scanner Z, Demographics W**Distribution Shifts:**- **Equipment**: Different scanners, protocols- **Demographics**: Age, ethnicity, disease prevalence- **Image quality**: Resolution, contrast, artifacts- **Annotation style**: Different radiologists**Case Study: Diabetic Retinopathy Detection**```Stanford model (2018):- Training: EyePACS dataset (US clinics)- AUC: 0.99 on test setDeployment to India:- Different imaging equipment- Different patient demographics- AUC dropped to 0.88 ← DANGEROUS!Solution with domain generalization:- Multi-center training- Domain randomization on images- Test-time adaptation- Final AUC: 0.96 (recovered 8 points)```**Why It Matters:**- ❤️ **Lives at stake**: Misdiagnosis can be fatal- 💰 **Cost**: Retraining per hospital is expensive- 🌍 **Access**: Enables deployment in resource-limited settings### 3. **Robotics: Sim-to-Real Transfer** 🤖**The Problem:**- Training in simulation: Fast, safe, cheap- Deployment in real world: Slow, dangerous, expensive**The Reality Gap:**| Aspect | Simulation | Real World ||--------|-----------|------------|| Physics | Simplified | Complex || Sensors | Perfect | Noisy || Lighting | Controlled | Variable || Textures | Synthetic | Natural || Dynamics | Deterministic | Stochastic |**Example: Robotic Grasping**```Naive Approach:- Train in sim: 99% success- Test in real: 23% success ❌With Domain Randomization:- Randomize: lighting, textures, object shapes, dynamics- Train in sim: 85% success (harder due to variation)- Test in real: 78% success ✅```**Techniques:**1. **Domain randomization**: Vary sim parameters2. **Adversarial training**: Make features domain-invariant3. **Fine-tuning**: Small amount of real data4. **Meta-learning**: Quick adaptation to real world**Success Stories:**- **OpenAI Dactyl** (2018): Rubik's cube solving, sim-to-real- **Google Brain** (2019): Robotic grasping, 96% success- **ANYmal** (2019): Quadruped locomotion, diverse terrains### 4. **Finance and Fraud Detection** 💳**The Challenge:**- Training: Historical data (normal market conditions)- Deployment: Market crashes, new fraud patterns, regulatory changes**Distribution Shifts:**- **Temporal**: Financial markets evolve- **Adversarial**: Fraudsters adapt to detection- **Regime changes**: Crises, policy changes**Example: Credit Card Fraud**```Standard model:- Train on 2020 fraud patterns- 2021: New fraud technique emerges- Detection rate drops from 94% to 67% ❌Continual OOD-robust model:- Trained on multiple time periods- Online adaptation- Anomaly detection component- Detection rate: 89% (maintained) ✅```### 5. **Natural Language Processing** 📝**Distribution Shifts:**- **Domain**: Medical ↔ Legal ↔ Social media- **Dialect**: AAVE vs Standard English- **Time**: Language evolves (slang, new terms)- **Formality**: Formal vs casual text**Example: Sentiment Analysis**```Training: Movie reviewsTest: Product reviewsStandard model: 88% → 72% accuracy ❌OOD-robust (multi-domain): 88% → 83% accuracy ✅```### Impact Metrics from Industry:| Company | Application | OOD Problem | Solution | Improvement ||---------|------------|-------------|----------|-------------|| Waymo | Self-driving | Weather shifts | Multi-domain training | -40% error rate || Google Health | Diabetic retinopathy | Scanner differences | Domain generalization | +8% AUC || OpenAI | Robotic manipulation | Sim-to-real | Domain randomization | +55% success || Stripe | Fraud detection | Evolving patterns | Continual learning | +12% detection |### Key Takeaways:✅ **OOD robustness is not optional** for real-world deployment✅ **Small accuracy drops can have huge consequences**✅ **Domain generalization prevents costly retraining**✅ **Meta-learning provides principled approach**

<a name='14'></a>## 14 - Summary and Conclusions<font color='blue'>**What you should remember:**✅ **OOD generalization** is critical for real-world ML deployment✅ **Distribution shift** occurs when $P_{\text{train}} \neq P_{\text{test}}$✅ **Types of shifts**: Covariate, label, concept drift, domain shift✅ **Standard ML fails** because it assumes IID data✅ **Domain randomization** exposes model to diverse training distributions✅ **Meta-learning** learns to generalize across distribution families✅ **Applications**: Autonomous vehicles, medicine, robotics, finance✅ **Robustness metrics** quantify performance under distribution shift</font>### Key Concepts Recap:**1. The Problem:**- Real-world ≠ training data- Standard ML overfits to training distribution- Failures can be catastrophic (safety, medicine, finance)**2. Approaches to OOD Robustness:**<table><tr>    <td><b>Approach</b></td>    <td><b>How It Works</b></td>    <td><b>Pros</b></td>    <td><b>Cons</b></td></tr><tr>    <td>Data Augmentation</td>    <td>Add noise, transforms</td>    <td>Simple, effective</td>    <td>Limited diversity</td></tr><tr>    <td>Domain Randomization</td>    <td>Random variations</td>    <td>More diversity</td>    <td>Needs many domains</td></tr><tr>    <td>Multi-Domain Training</td>    <td>Train on K domains</td>    <td>Learns from diversity</td>    <td>Requires domain labels</td></tr><tr>    <td>Meta-Learning</td>    <td>Optimize for generalization</td>    <td>Principled, effective</td>    <td>Computationally expensive</td></tr><tr>    <td>Domain Adaptation</td>    <td>Adapt to specific target</td>    <td>Best if target known</td>    <td>Needs target data</td></tr></table>**3. When to Use Each:**✅ **Data Augmentation**: Always (cheap, easy wins)✅ **Domain Randomization**: When you can simulate variations (robotics, vision)✅ **Meta-Learning**: When you have multiple related domains✅ **Domain Adaptation**: When you have access to unlabeled target data### Practical Guidelines:**During Development:**1. **Identify potential shifts**: What will change at deployment?2. **Collect diverse data**: Multiple domains if possible3. **Use augmentation**: Simulate realistic variations4. **Evaluate on OOD**: Test on held-out domains**During Deployment:**1. **Monitor distribution**: Detect when inputs drift2. **Test-time adaptation**: Update model on target domain3. **Confidence calibration**: Reject very OOD examples4. **Graceful degradation**: Fail safely when uncertain**Metrics to Track:**- Accuracy on ID (in-distribution) test- Accuracy on OOD variants- **OOD gap**: ID accuracy - OOD accuracy ← **Minimize this!**- Calibration error (confidence vs actual accuracy)### 📚 Essential References:1. **IRM**: [Invariant Risk Minimization](https://arxiv.org/abs/1907.02893)2. **DomainBed**: [In Search of Lost Domain Generalization](https://arxiv.org/abs/2007.01434)3. **MLDG**: [Learning to Generalize: Meta-Learning for Domain Generalization](https://arxiv.org/abs/1710.03463)4. **DANN**: [Domain-Adversarial Training of Neural Networks](https://arxiv.org/abs/1505.07818)5. **Ben-David et al.**: [Theory of Domain Adaptation](https://link.springer.com/article/10.1007/s10994-009-5152-4)### 🚀 Next Steps:1. Implement on your own domain2. Try advanced methods: IRM, CORAL, GroupDRO3. Explore test-time adaptation techniques4. Combine with meta-learning for best results5. Always evaluate on OOD data before deployment!---## 🎉 Congratulations!You now understand **OOD generalization**, one of the most critical challenges for deploying AI!**You've learned**:- Why distribution shift matters- Types of shifts and their impacts- Techniques for improving robustness- Real-world applications and case studies- How to evaluate and monitor OOD performance**You're ready to**:- Build robust models for real-world deployment- Handle distribution shifts systematically- Apply domain generalization techniques- Deploy AI safely in critical domains!**Final Tutorial**: Tutorial 09 brings everything together in a comprehensive project! 🎯